# Purpose:
- Compare cell matching between
    - ROICat
    - Registering to cortical z-stack

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

In [25]:
# Load fov to cortical z-stack matching table
using_stack_df = pd.read_csv('/root/capsule/scratch/glm/thyme_fov_to_czstack_match_table.csv')
using_stack_df['session_key'] = using_stack_df.session_name.apply(lambda x: '_'.join(x.split('_')[1:3]))
using_stack_df['session_roi_name'] = using_stack_df.apply(lambda x: f'{"_".join(x.session_key.split("_")[-2:])}_{x.plane}_{x.session_roi_id:04}', axis=1)

In [26]:
# Filter duplicated rois based on iou
temp_series = using_stack_df.groupby('session_roi_name').size()
temp_series = temp_series[temp_series > 1]
print(f'Expected number of rows to remove: {temp_series.sum() - len(temp_series)}')
# print(temp_series.max())
duplicated_roi_names = temp_series.index.values
print(f'Number of duplicated ROIs: {len(duplicated_roi_names)}')
duplicated_roi_df = using_stack_df[using_stack_df.session_roi_name.isin(duplicated_roi_names)].copy()
print(len(duplicated_roi_df))
num_rows_to_remove = len(duplicated_roi_df) - len(duplicated_roi_names)
print(f'Number of rows to remove: {num_rows_to_remove}')
print(f'Expected rows after removal: {len(using_stack_df) - num_rows_to_remove}')
inds_to_remove = duplicated_roi_df.index.values
duplicated_roi_df.sort_values('max_iou', ascending=False, inplace=True)
duplicated_roi_df.drop_duplicates('session_roi_name', keep='first', inplace=True)
print(len(duplicated_roi_df))
inds_to_keep = duplicated_roi_df.index.values
inds_to_remove = np.setdiff1d(inds_to_remove, inds_to_keep)
using_stack_df.drop(inds_to_remove, inplace=True)
print(f'Rows after removal: {len(using_stack_df)}')
assert (using_stack_df.groupby('session_roi_name').size()>1).sum() == 0

Expected number of rows to remove: 97
Number of duplicated ROIs: 96
193
Number of rows to remove: 97
Expected rows after removal: 9124
96
Rows after removal: 9124


In [27]:
roicat_df.valid_roi.sum()

5226

In [28]:
roicat_df = pd.read_pickle('/root/capsule/scratch/glm/roicat_df_736963.pkl')
print(len(roicat_df))
valid_roicat_df = roicat_df.query('valid_roi').copy()
print(len(valid_roicat_df))
print(valid_roicat_df.unique_roi_name.nunique())
print(valid_roicat_df.groupby('unique_roi_name').size().sum())

7069
5226
934
5226


In [29]:
# Check the number of rows in using_stack_df that are in valid_roicat_df
session_keys_in_roicat = valid_roicat_df.session_key.unique()
nis_using_stack_df = using_stack_df[using_stack_df.session_key.isin(session_keys_in_roicat)].copy() # natural image sessions
print(len(nis_using_stack_df))
print(len(nis_using_stack_df)/len(valid_roicat_df))
# using_stack_df is not fully filtered by valid rois
# (so there are some rois that are not in valid_roicat_df)

3978
0.7611940298507462


In [30]:
using_stack_df

,fov_id,cz_stack_id,max_iou,session_name,plane,session_roi_id,session_key,session_roi_name
0,VISp_0_74,342,0.67,multiplane-ophys_736963_2024-08-27_08-51-49_pr...,VISp_0,74,736963_2024-08-27,736963_2024-08-27_VISp_0_0074
1,VISp_0_10,344,0.70,multiplane-ophys_736963_2024-08-27_08-51-49_pr...,VISp_0,10,736963_2024-08-27,736963_2024-08-27_VISp_0_0010
2,VISp_0_55,346,0.27,multiplane-ophys_736963_2024-08-27_08-51-49_pr...,VISp_0,55,736963_2024-08-27,736963_2024-08-27_VISp_0_0055
3,VISp_0_24,350,0.67,multiplane-ophys_736963_2024-08-27_08-51-49_pr...,VISp_0,24,736963_2024-08-27,736963_2024-08-27_VISp_0_0024
4,VISp_0_22,352,0.44,multiplane-ophys_736963_2024-08-27_08-51-49_pr...,VISp_0,22,736963_2024-08-27,736963_2024-08-27_VISp_0_0022
...,...,...,...,...,...,...,...,...
9216,VISp_7_18,780,0.75,multiplane-ophys_736963_2024-08-26_09-54-51_pr...,VISp_7,18,736963_2024-08-26,736963_2024-08-26_VISp_7_0018
9217,VISp_7_10,785,0.23,multiplane-ophys_736963_2024-08-26_09-54-51_pr...,VISp_7,10,736963_2024-08-26,736963_2024-08-26_VISp_7_0010
9218,VISp_7_68,789,0.17,multiplane-ophys_736963_2024-08-26_09-54-51_pr...,VISp_7,68,736963_2024-08-26,736963_2024-08-26_VISp_7_0068
9219,VISp_7_12,790,0.59,multiplane-ophys_736963_2024-08-26_09-54-51_pr...,VISp_7,12,736963_2024-08-26,736963_2024-08-26_VISp_7_0012


In [31]:
# How about per session?
# Look at it after merging
reduced_valid_roicat_df = valid_roicat_df.reset_index()[['session_roi_name', 'unique_roi_name', 'session_key']].copy()
reduced_using_stack_df = using_stack_df[['session_roi_name', 'cz_stack_id']].copy()
comp_df = reduced_valid_roicat_df.merge(reduced_using_stack_df,
                                        on='session_roi_name',
                                        how='left')
comp_df


,session_roi_name,unique_roi_name,session_key,cz_stack_id
0,736963_2024-08-09_VISp_0_0004,736963_VISp_0_0013,736963_2024-08-09,NaN
1,736963_2024-08-09_VISp_0_0005,736963_VISp_0_0088,736963_2024-08-09,379.0
2,736963_2024-08-09_VISp_0_0006,736963_VISp_0_0092,736963_2024-08-09,439.0
3,736963_2024-08-09_VISp_0_0007,736963_VISp_0_0101,736963_2024-08-09,363.0
4,736963_2024-08-09_VISp_0_0008,736963_VISp_0_0098,736963_2024-08-09,361.0
...,...,...,...,...
5221,736963_2024-08-06_VISp_7_0093,736963_VISp_7_0071,736963_2024-08-06,NaN
5222,736963_2024-08-06_VISp_7_0095,736963_VISp_7_0084,736963_2024-08-06,789.0
5223,736963_2024-08-06_VISp_7_0096,736963_VISp_7_0088,736963_2024-08-06,NaN
5224,736963_2024-08-06_VISp_7_0097,736963_VISp_7_0080,736963_2024-08-06,NaN


In [32]:
# proportion dropped when matching to cortical z-stack
prop_missing_from_stack = comp_df.cz_stack_id.isna().mean()
print(f'Proportion of ROIs missing from cortical z-stack: {prop_missing_from_stack}')
num_rois_per_session = comp_df.groupby('session_key').size()
num_missing_rois_per_session = comp_df[comp_df.cz_stack_id.isna()].groupby('session_key').size()
prop_missing_rois_per_session = num_missing_rois_per_session/num_rois_per_session
prop_missing_rois_per_session


Proportion of ROIs missing from cortical z-stack: 0.34998086490623803


session_key
736963_2024-07-24    0.344538
736963_2024-07-26    0.328829
736963_2024-07-29    0.329218
736963_2024-07-30    0.361377
736963_2024-08-01    0.347458
736963_2024-08-05    0.329621
736963_2024-08-06    0.372745
736963_2024-08-07    0.329384
736963_2024-08-09    0.334052
736963_2024-08-12    0.364017
736963_2024-08-13    0.397661
dtype: float64

In [33]:
final_comp_df = comp_df[comp_df.cz_stack_id.notna()].copy()
print(f'Num unique roi names from roicat: {final_comp_df.unique_roi_name.nunique()}')
print(f'Num rois from cortical z-stack: {final_comp_df.cz_stack_id.nunique()}')


Num unique roi names from roicat: 500
Num rois from cortical z-stack: 436


In [34]:
final_comp_df.groupby('unique_roi_name')['cz_stack_id'].nunique().value_counts()

cz_stack_id
1    487
2     13
Name: count, dtype: int64

In [35]:
final_comp_df.groupby('cz_stack_id')['unique_roi_name'].nunique().value_counts()

unique_roi_name
1     374
2      58
3       2
6       1
11      1
Name: count, dtype: int64

In [36]:
from_zstack_to_roicat = final_comp_df.groupby('cz_stack_id')['unique_roi_name'].nunique()
from_zstack_to_roicat[from_zstack_to_roicat > 2]

cz_stack_id
346.0     6
532.0    11
626.0     3
684.0     3
Name: unique_roi_name, dtype: int64

In [37]:
np.sort(final_comp_df.session_key.unique())

array(['736963_2024-07-24', '736963_2024-07-26', '736963_2024-07-29',
       '736963_2024-07-30', '736963_2024-08-01', '736963_2024-08-05',
       '736963_2024-08-06', '736963_2024-08-07', '736963_2024-08-09',
       '736963_2024-08-12', '736963_2024-08-13'], dtype=object)

In [38]:
# What if I reduce to the ones in FNN session?
fnn_session_keys = ['736963_2024-08-06', '736963_2024-08-07', '736963_2024-08-09']
fnn_comp_df = final_comp_df[final_comp_df.session_key.isin(fnn_session_keys)].copy()
display(fnn_comp_df.groupby('unique_roi_name')['cz_stack_id'].nunique().value_counts())
display(fnn_comp_df.groupby('cz_stack_id')['unique_roi_name'].nunique().value_counts())


cz_stack_id
1    404
2      5
Name: count, dtype: int64

unique_roi_name
1    335
2     38
3      1
Name: count, dtype: int64

In [39]:
# save the final comp_df
save_dir = Path('/root/capsule/scratch/roi_matching_comparison/736963')
save_dir.mkdir(exist_ok=True, parents=True)
final_comp_df.to_csv(save_dir /'736963_roi_matching_comp_df_v1.csv', index=False)

# Conclusion:
- There is multi matching examples.
- Check them visually.
- In a separate capsule. (CTL QC)
    - I also need the result from this capsule (or this notebook)

In [40]:
merged_df = pd.read_csv('/root/capsule/scratch/glm/Thyme_merged_roi_ids_with_cluster_ids.csv')